# Решения: сравнение признаков

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def find_listings_csv() -> Path:
    for p in (Path('listings.csv'), Path('../../data/listings.csv'), Path('../data/listings.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('listings.csv не найден')


LISTINGS_PATH = find_listings_csv()
df = pd.read_csv(LISTINGS_PATH)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


## Урок. 1–3. eval_feature

In [ ]:
def eval_feature(feature_name: str, random_state: int = 42):
    X = df[[feature_name]]
    y = df['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    pred = LinearRegression().fit(X_tr, y_tr).predict(X_te)
    return mean_squared_error(y_te, pred), r2_score(y_te, pred)


mse_d, r2_d = eval_feature('accommodates')
mse_r, r2_r = eval_feature('number_of_reviews')
print('accommodates', round(mse_d, 2), round(r2_d, 3))
print('number_of_reviews', round(mse_r, 2), round(r2_r, 3))

## Урок. 4. Таблица

In [ ]:
compare_table = pd.DataFrame([
    {'feature': 'accommodates', 'mse': mse_d, 'r2': r2_d},
    {'feature': 'number_of_reviews', 'mse': mse_r, 'r2': r2_r},
])
print(compare_table)

## Урок. 5–6. BETTER_FEATURE / WHY

In [ ]:
BETTER_FEATURE = 'accommodates' if r2_d >= r2_r else 'number_of_reviews'
WHY = (
    f'По R² на test лучше {BETTER_FEATURE} '
    f'(accommodates R²={r2_d:.3f}, number_of_reviews R²={r2_r:.3f}). '
    'Для отчёта фиксируем один признак и те же seed/test_size.'
)
print(BETTER_FEATURE)
print(WHY)

## Урок. 7–8. Multi-feature обзор

In [ ]:
y = df['price']
X1 = df[['accommodates']]
X2 = df[['accommodates', 'number_of_reviews']]
X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y, test_size=0.2, random_state=42)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.2, random_state=42)
r2_one = r2_score(y1_te, LinearRegression().fit(X1_tr, y1_tr).predict(X1_te))
pred2 = LinearRegression().fit(X2_tr, y2_tr).predict(X2_te)
r2_two = r2_score(y2_te, pred2)
two_better = r2_two > r2_one
MULTI_NOTE = (
    'Второй признак стоит пробовать, но решение — по метрике на test: '
    'лишний столбец может не помочь или чуть ухудшить R².'
)
print('one', round(r2_one, 3), 'two', round(r2_two, 3), 'two_better', two_better)
print(MULTI_NOTE)

## ДЗ. 1. Таблица

In [ ]:
rows = []
for f in ('accommodates', 'number_of_reviews'):
    m, r = eval_feature(f)
    rows.append({'feature': f, 'mse': m, 'r2': r})
table = pd.DataFrame(rows)
print(table)

## ДЗ. 2. is_entire

In [ ]:
df2 = df.copy()
df2['is_entire'] = (df2['room_type'] == 'entire').astype(int)

def eval_feature_df(frame, feature_name: str, random_state: int = 42):
    X = frame[[feature_name]]
    y = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    pred = LinearRegression().fit(X_tr, y_tr).predict(X_te)
    return mean_squared_error(y_te, pred), r2_score(y_te, pred)


mse_c, r2_c = eval_feature_df(df2, 'is_entire')
beats_accommodates = r2_c > r2_d
print(r2_c, beats_accommodates)

## ДЗ. 3. MSE two + KEEP_ONE

In [ ]:
mse_two = mean_squared_error(y2_te, pred2)
KEEP_ONE_NOTE = (
    'Если второй признак не улучшает R² на test или усложняет интерпретацию — '
    'в сдачу оставляем один столбец.'
)
print(round(mse_two, 2), KEEP_ONE_NOTE)